# CLERNet — Aggregation, Selection, and Metrics

Loads the predictions JSON produced by `CLERNet_exec.ipynb` and applies:

1. **Aggregation** — cumulative sum of per-step per-fluent activations, with
   domain-specific mutex exclusion to suppress physically incompatible fluents.
2. **Selection** — at each step the candidate hypothesis with the highest
   aggregated score is selected as the predicted goal.
3. **Metrics** — three scores are reported:
   - **RF (Ranked First)**: fraction of steps across all plans where the correct
     hypothesis is ranked first (step-weighted average).
   - **CV (Convergence)**: fraction of steps from the first step at which the model
     selects the correct hypothesis *and never switches away* until the end.
   - **Accuracy**: same as RF but reported as a plain average (unweighted).


## 1. Configuration


In [ ]:
import os

REPO_ROOT        = os.path.abspath(os.path.join(os.getcwd(), ".."))  # CLERNet_public/

# Path to the JSON written by CLERNet_exec.ipynb
PREDICTIONS_JSON = os.path.join(REPO_ROOT, "results", "blocksworld_predictions.json")

# Directory with goal-vocabulary pickle files (one sub-folder per domain)
DICT_DIR         = os.path.join(REPO_ROOT, "data", "custom_dataset")

DOMAIN           = "blocksworld"  # must match the predictions JSON

# Per-step activations below TAU_NOISE are zeroed before accumulation
TAU_NOISE  = 0.05
# Activations above TAU_MUTEX trigger mutex exclusion of conflicting fluents
TAU_MUTEX  = 0.40


## 2. Imports


In [ ]:
import json
import pickle
import warnings
import numpy as np
from os.path import join


## 3. Load Resources


In [ ]:
with open(PREDICTIONS_JSON) as f:
    predictions_data = json.load(f)

dict_path = join(DICT_DIR, DOMAIN, "dizionario_goal")
with open(dict_path, "rb") as f:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        goal_vocab = pickle.load(f)

print(f"Instances loaded : {len(predictions_data)}")
print(f"Goal vocab size  : {len(goal_vocab)}")


## 4. Mutex Helper Functions

Mutex exclusion prevents physically incompatible fluents from accumulating
simultaneously. For example in Blocksworld, if `ON A B` is confidently predicted,
any other `ON A ?` or `ON ? B` predicate is zeroed out in the running sum.


In [ ]:
def _get_fluent_key(fluent_idx: int, goal_vocab: dict):
    """Reverse-lookup: goal index → predicate string."""
    for k, v in goal_vocab.items():
        if int(np.argmax(v)) == fluent_idx:
            return k
    return None


def _mutex_generic(fluent_idx: int, goal_vocab: dict) -> set:
    """Mutex by predicate prefix (works for Logistics, Driverlog, Zenotravel)."""
    fluent = _get_fluent_key(fluent_idx, goal_vocab)
    prefix = fluent.rsplit(' ', 1)[0]
    return {int(np.argmax(v)) for k, v in goal_vocab.items() if k.startswith(prefix)}


def _mutex_depots(fluent_idx: int, goal_vocab: dict) -> set:
    """Mutex for Depots: match both predicate prefix and last argument."""
    fluent = _get_fluent_key(fluent_idx, goal_vocab)
    parts  = fluent.rsplit(' ', 1)
    return {
        int(np.argmax(v)) for k, v in goal_vocab.items()
        if k.startswith(parts[0]) or k.endswith(parts[-1])
    }


def _mutex_satellite(fluent_idx: int, goal_vocab: dict) -> set:
    """Mutex for Satellite: only POINTING predicates are mutually exclusive."""
    fluent = _get_fluent_key(fluent_idx, goal_vocab)
    if not fluent.startswith("pointing"):
        return set()
    prefix = fluent.rsplit(' ', 1)[0]
    return {int(np.argmax(v)) for k, v in goal_vocab.items() if k.startswith(prefix)}


def _mutex_blocksworld(fluent_idx: int, goal_vocab: dict) -> set:
    """Mutex for Blocksworld: a block can occupy only one position at a time."""
    fluent = _get_fluent_key(fluent_idx, goal_vocab)
    parts  = fluent.split(' ')
    mutex  = set()
    for k, v in goal_vocab.items():
        obj = k.split(' ')
        if parts[0] in ('on', 'ontable', 'on-table'):
            if obj[0] == 'on' and len(obj) >= 3:
                if obj[1] == parts[1] or (len(parts) > 2 and obj[2] == parts[2]):
                    mutex.add(int(np.argmax(v)))
            if obj[0] in ('on-table', 'ontable') and len(parts) > 2 and obj[-1] == parts[2]:
                mutex.add(int(np.argmax(v)))
        if parts[0] == 'clear' and obj[0] == 'on' and len(obj) >= 3 and obj[2] == parts[1]:
            mutex.add(int(np.argmax(v)))
    return mutex


_MUTEX_FN = {
    "depots":      _mutex_depots,
    "satellite":   _mutex_satellite,
    "blocksworld": _mutex_blocksworld,
}

def get_mutex_fn(domain: str):
    """Return the mutex function for the given domain."""
    return _MUTEX_FN.get(domain, _mutex_generic)


## 5. Aggregation and Selection

At each observation step:
1. Per-step raw activations below `TAU_NOISE` are zeroed.
2. The filtered activations are added to the **running sum**.
3. Any fluent above `TAU_MUTEX` triggers mutex exclusion of conflicting fluents.
4. Each candidate hypothesis is scored as the sum of the running values at its
   constituent fluent indices.
5. The hypothesis with the highest score is **selected** as the current prediction.


In [ ]:
mutex_fn = get_mutex_fn(DOMAIN)
instance_results = []  # one dict per instance

for instance_name, data in predictions_data.items():
    obs_len         = data["obs_len"]
    possible_goals  = data["possible_goals_indices"]   # list of list of ints
    correct_hyp_idx = data["correct_hyp_idx"]
    step_preds      = data["predictions"]              # (obs_len, n_goals)

    running_sum = np.zeros(len(goal_vocab))
    step_correct = []  # 1 if selected == correct at this step, else 0

    for t, raw_preds in enumerate(step_preds):
        preds = np.array(raw_preds)

        # 1. Noise suppression
        preds_filtered = np.where(preds > TAU_NOISE, preds, 0.0)

        # 2. Accumulate
        running_sum += preds_filtered

        # 3. Mutex exclusion
        high_idx = [i for i, p in enumerate(preds_filtered) if p > TAU_MUTEX]
        for hi in high_idx:
            for m in mutex_fn(hi, goal_vocab):
                if m not in high_idx:
                    running_sum[m] = 0.0

        # 4. Score hypotheses
        scores      = [float(np.sum(running_sum[hyp])) for hyp in possible_goals]
        selected    = int(np.argmax(scores))
        step_correct.append(1 if selected == correct_hyp_idx else 0)

    instance_results.append({
        "instance":       instance_name,
        "obs_len":        obs_len,
        "correct_hyp":    correct_hyp_idx,
        "step_correct":   step_correct,
    })


## 6. Metrics: RF, CV, and Accuracy

**RF (Ranked First)** — For each plan, count the steps where the correct hypothesis
is ranked first, divided by the plan length. Average over all plans.

**CV (Convergence)** — For each plan, find the earliest step `t*` from which the
model always selects the correct hypothesis until the end (i.e., the suffix
`step_correct[t*:]` is all-ones). CV = `(plan_length - t*) / plan_length`.
If the model never converges, CV = 0 for that plan.

**Accuracy** — Fraction of steps (unweighted) where the correct hypothesis is ranked
first, averaged over all plans.


In [ ]:
def compute_cv(step_correct: list) -> float:
    """Convergence score: fraction of plan covered by the longest correct suffix.

    Finds the earliest step t* such that step_correct[t*:] is all-ones and returns
    (n - t*) / n.  Returns 0.0 if the model never selects the correct hypothesis
    without backtracking.
    """
    n = len(step_correct)
    if n == 0:
        return 0.0
    # Walk backwards to find where the trailing correct streak starts
    t_star = n
    for t in range(n - 1, -1, -1):
        if step_correct[t] == 1:
            t_star = t
        else:
            break
    return (n - t_star) / n


rf_list, cv_list, acc_list = [], [], []

print(f"{'Instance':<55}  RF     CV     Acc")
print("-" * 75)

for r in instance_results:
    sc   = r["step_correct"]
    n    = r["obs_len"]
    rf   = sum(sc) / n if n else 0.0
    cv   = compute_cv(sc)
    acc  = sum(sc) / n if n else 0.0  # same as RF here (unweighted)

    rf_list.append(rf)
    cv_list.append(cv)
    acc_list.append(acc)

    print(f"{r['instance'][:55]:<55}  {rf:.3f}  {cv:.3f}  {acc:.3f}")

print()
print(f"=== Summary over {len(instance_results)} instances ===")
print(f"  Avg RF  (Ranked First)  : {sum(rf_list)  / len(rf_list):.4f}")
print(f"  Avg CV  (Convergence)   : {sum(cv_list)  / len(cv_list):.4f}")
print(f"  Avg Acc                 : {sum(acc_list) / len(acc_list):.4f}")
